In [ ]:
import pandas as pd
import numpy as np
import os
from IPython.display import display, HTML, Image, Markdown
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from xgboost import XGBRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.utils import resample
import warnings

warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold

!rm -rf /kaggle/working/*
    
print("#"*150)
train=pd.read_csv("/kaggle/input/winners-will-will-exciting-prizes/train.csv")
print(f"Train Data Shape: {train.shape}")
print(f"Train Data INFO: {train.info()}")
print(f"Check out NULL data ON Train Data: {train.isnull().sum()}")
print(f"Value Counts: {train['quality'].value_counts()}")
# mapping = {3: 0, 4: 1, 5: 2, 6: 3, 7: 4, 8: 5}

# # Apply mapping to the 'quality' column
# train['quality'] = train['quality'].replace(mapping)

display(train.head())
print("#"*150)

test=pd.read_csv("/kaggle/input/winners-will-will-exciting-prizes/test.csv")
print(f"Test Data Shape: {test.shape}")
print(f"Test Data INFO: {test.info()}")
print(f"Check out NULL data ON Test Data: {test.isnull().sum()}")
display(test.head())
print("#"*150)

In [ ]:
# train_major = train[train['quality'] == 5.0]
# max_count = len(train_major)

# dfs = []

# for label in train['quality'].unique():
#     df_label = train[train['quality'] == label]
#     if len(df_label) < max_count:
#         df_label_upsampled = resample(df_label, replace=True, n_samples=max_count, random_state=42)
#         dfs.append(df_label_upsampled)
#     else:
#         dfs.append(df_label)

# train = pd.concat(dfs).sample(frac=1, random_state=42).reset_index(drop=True)

# print(train['quality'].value_counts())


In [ ]:
# import lightgbm as lgb
# from sklearn.model_selection import KFold
# from sklearn.metrics import accuracy_score, f1_score
# import numpy as np
# import pandas as pd

# X = train.drop(['id', 'quality'], axis=1)
# y = train['quality']
# X_test = test.drop('id', axis=1)

# cv_acc = []
# cv_f1_macro = []
# kf = KFold(n_splits=10, shuffle=True, random_state=42)

# for fold, (train_idx, val_idx) in enumerate(kf.split(X, y), 1):
#     X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
#     y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
#     lgb_clf = lgb.LGBMClassifier(
#         n_estimators=2000,
#         max_depth=10,
#         learning_rate=0.005,
#         random_state=42,
#         n_jobs=-1,
#         device="gpu",
#         verbosity=-1
#     )
    
#     lgb_clf.fit(X_train, y_train,eval_set=[(X_val, y_val)])
    
#     val_pred = lgb_clf.predict(X_val)
    
#     acc = accuracy_score(y_val, val_pred)
#     f1_macro = f1_score(y_val, val_pred, average='macro')
    
#     cv_acc.append(acc)
#     cv_f1_macro.append(f1_macro)
    
#     print(f"Fold {fold} — Accuracy: {acc:.4f}, F1 Macro: {f1_macro:.4f}")

# print(f"\nMean CV Accuracy: {np.mean(cv_acc):.4f} ± {np.std(cv_acc):.4f}")
# print(f"Mean CV F1 Macro: {np.mean(cv_f1_macro):.4f} ± {np.std(cv_f1_macro):.4f}")

In [ ]:
# # # Train on full data
# lgb_clf.fit(X, y)
# test_pred = lgb_clf.predict(X_test)

# submission = pd.DataFrame({'id': test['id'], 'quality': test_pred})
# submission.to_csv("submission_rf_classifier.csv", index=False)
# print("Submission file created: submission_rf_classifier.csv")

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
from sklearn.utils import resample

# -------------------------- CONFIG --------------------------
DATA_PATH = "/kaggle/input/winners-will-will-exciting-prizes"
TRAIN_CSV = os.path.join(DATA_PATH, "train.csv")
TEST_CSV  = os.path.join(DATA_PATH, "test.csv")
SUBMISSION_CSV = "submission_ensemble.csv"

N_FOLDS = 5
SEED = 42
np.random.seed(SEED)

# -------------------------- MODEL PARAMS --------------------------

# Decision Tree
DT_PARAMS = {
    'max_depth': 8,
    'min_samples_split': 20,
    'min_samples_leaf': 10,
    'random_state': SEED
}

# Random Forest
RF_PARAMS = {
    'n_estimators': 600,
    'max_depth': 12,
    'min_samples_split': 10,
    'min_samples_leaf': 5,
    'max_features': 'sqrt',
    'random_state': SEED,
    'n_jobs': -1
}

# LightGBM (GPU)
LGB_PARAMS = {
    'objective': 'multiclass', 'num_class': 6, 'metric': 'multi_logloss',
    'learning_rate': 0.05, 'max_depth': 6, 'num_leaves': 64,
    'feature_fraction': 0.85, 'bagging_fraction': 0.85, 'bagging_freq': 1,
    'lambda_l1': 0.1, 'lambda_l2': 0.1, 'min_child_samples': 20,
    'verbosity': -1, 'random_state': SEED,
    'device': 'gpu', 'early_stopping_rounds': 50
}

# -------------------------- LOAD DATA --------------------------
train = pd.read_csv(TRAIN_CSV)
test  = pd.read_csv(TEST_CSV)

# -------------------------- GLOBAL UPSAMPLING --------------------------
train_major = train[train['quality'] == 5.0]
max_count = len(train_major)
dfs = []

for label in train['quality'].unique():
    df_label = train[train['quality'] == label]
    if len(df_label) < max_count:
        df_label_upsampled = resample(df_label, replace=True, n_samples=max_count, random_state=42)
        dfs.append(df_label_upsampled)
    else:
        dfs.append(df_label)

train_upsampled = pd.concat(dfs).sample(frac=1, random_state=42).reset_index(drop=True)

# -------------------------- PREPARE FEATURES --------------------------
X = train_upsampled.drop(columns=['id', 'quality'])
y = train_upsampled['quality']
y_mapped = (y - 3).astype(int)
X_test = test.drop(columns=['id'])

# -------------------------- CROSS-VALIDATION --------------------------
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_dt = np.zeros(len(X))
oof_rf = np.zeros(len(X))
oof_lgb = np.zeros(len(X))
test_dt = np.zeros(len(X_test))
test_rf = np.zeros(len(X_test))
test_lgb = np.zeros((len(X_test), 6))  # Probabilities
cv_scores = []
scaler = StandardScaler()

print("Starting 5-Fold Ensemble CV (DT + RF + LGB)...\n")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_mapped)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train_mapped, y_val_mapped = y_mapped.iloc[train_idx], y_mapped.iloc[val_idx]
    
    # Scale for LGB
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled   = scaler.transform(X_val)
    X_test_scaled  = scaler.transform(X_test)
    
    # --- Decision Tree ---
    dt = DecisionTreeClassifier(**DT_PARAMS)
    dt.fit(X_train, y_train_mapped)
    val_pred_dt = dt.predict(X_val)
    test_pred_dt = dt.predict(X_test)
    oof_dt[val_idx] = val_pred_dt
    test_dt += test_pred_dt / N_FOLDS
    
    # --- Random Forest ---
    rf = RandomForestClassifier(**RF_PARAMS)
    rf.fit(X_train, y_train_mapped)
    val_pred_rf = rf.predict(X_val)
    test_pred_rf = rf.predict(X_test)
    oof_rf[val_idx] = val_pred_rf
    test_rf += test_pred_rf / N_FOLDS
    
    # --- LightGBM ---
    lgb_train = lgb.Dataset(X_train_scaled, label=y_train_mapped)
    lgb_val   = lgb.Dataset(X_val_scaled, label=y_val_mapped, reference=lgb_train)
    lgb_model = lgb.train(LGB_PARAMS, lgb_train, valid_sets=[lgb_val])
    val_pred_lgb_proba = lgb_model.predict(X_val_scaled)
    val_pred_lgb = np.argmax(val_pred_lgb_proba, axis=1)
    oof_lgb[val_idx] = val_pred_lgb
    test_lgb += lgb_model.predict(X_test_scaled) / N_FOLDS
    
    # --- Ensemble OOF (average probabilities) ---
    # Convert DT/RF to one-hot probabilities
    dt_proba = np.eye(6)[val_pred_dt]
    rf_proba = np.eye(6)[val_pred_rf]
    ensemble_proba = (dt_proba + rf_proba + val_pred_lgb_proba) / 3
    ensemble_pred = np.argmax(ensemble_proba, axis=1)
    
    # Score
    fold_f1 = f1_score(y_val_mapped, ensemble_pred, average='macro')
    cv_scores.append(fold_f1)
    print(f"Fold {fold+1} | Macro-F1: {fold_f1:.5f}")

print(f"\nCV Macro-F1: {np.mean(cv_scores):.5f} ± {np.std(cv_scores):.5f}")

# -------------------------- FINAL ENSEMBLE PREDICTION --------------------------
# Convert DT/RF test preds to probabilities
dt_test_proba = np.eye(6)[test_dt.astype(int)]
rf_test_proba = np.eye(6)[test_rf.astype(int)]
final_proba = (dt_test_proba + rf_test_proba + test_lgb) / 3
final_labels = np.argmax(final_proba, axis=1)
final_preds = final_labels + 3

# -------------------------- SAVE SUBMISSION --------------------------
submission = pd.DataFrame({"id": test["id"], "quality": final_preds})
submission.to_csv(SUBMISSION_CSV, index=False)
print(f"Submission saved: {SUBMISSION_CSV}")

In [ ]:
submission["quality"].value_counts()

In [ ]:
sub = pd.read_csv('/kaggle/working/submission_ensemble.csv')
print(sub.head())
print(sub.info())
print("Shape:", sub.shape)
print("Unique quality:", sorted(sub['quality'].unique()))
print("Any NaNs?", sub.isnull().sum().sum())
print("IDs range:", sub['id'].min(), "to", sub['id'].max())

In [ ]:
# import pandas as pd
# import numpy as np
# from sklearn.linear_model import Ridge
# from sklearn.metrics import mean_squared_error, mean_absolute_error, f1_score, accuracy_score
# import warnings
# warnings.filterwarnings('ignore')

# train_df = pd.read_csv("/kaggle/input/winners-will-will-exciting-prizes/train.csv")
# test_df = pd.read_csv("/kaggle/input/winners-will-will-exciting-prizes/test.csv")
# sample_sub_df = pd.read_csv("/kaggle/input/winners-will-will-exciting-prizes/sample_submission.csv")

# def create_features(df):
#     df_new = df.copy()
#     df_new['acid_ratio'] = df_new['fixed acidity'] / (df_new['volatile acidity'] + 1e-5)
#     df_new['sulfur_ratio'] = df_new['free sulfur dioxide'] / (df_new['total sulfur dioxide'] + 1e-5)
#     df_new['sulfur_diff'] = df_new['total sulfur dioxide'] - df_new['free sulfur dioxide']
#     df_new['acid_sugar_interaction'] = df_new['citric acid'] * df_new['residual sugar']
#     df_new['alcohol_density'] = df_new['alcohol'] * df_new['density']
#     df_new['pH_alcohol'] = df_new['pH'] * df_new['alcohol']
#     df_new['sulphates_alcohol'] = df_new['sulphates'] * df_new['alcohol']
#     df_new['acid_pH'] = df_new['fixed acidity'] * df_new['pH']
#     df_new['chlorides_sulphates'] = df_new['chlorides'] * df_new['sulphates']
#     df_new['total_acidity'] = df_new['fixed acidity'] + df_new['volatile acidity'] + df_new['citric acid']
#     df_new['total_sulfur_dioxide_log'] = np.log1p(df_new['total sulfur dioxide'])
#     df_new['free_sulfur_dioxide_log'] = np.log1p(df_new['free sulfur dioxide'])
#     df_new['log_residual_sugar'] = np.log1p(df_new['residual sugar'])
#     df_new['log_chlorides'] = np.log1p(df_new['chlorides'])
#     df_new['log_volatile_acidity'] = np.log1p(df_new['volatile acidity'])
#     df_new['alcohol_squared'] = df_new['alcohol'] ** 2
#     df_new['sulphates_squared'] = df_new['sulphates'] ** 2
#     df_new['pH_squared'] = df_new['pH'] ** 2
#     df_new['density_squared'] = df_new['density'] ** 2
#     df_new['alcohol_category'] = pd.cut(df_new['alcohol'], bins=5, labels=False)
#     df_new['pH_category'] = pd.cut(df_new['pH'], bins=5, labels=False)
#     df_new['density_category'] = pd.cut(df_new['density'], bins=5, labels=False)
#     df_new['sulphates_category'] = pd.cut(df_new['sulphates'], bins=5, labels=False)
#     df_new['alcohol_cube'] = df_new['alcohol'] ** 3
#     df_new['sqrt_alcohol'] = np.sqrt(df_new['alcohol'])
#     df_new['sqrt_sulphates'] = np.sqrt(df_new['sulphates'])
#     df_new['alcohol_per_density'] = df_new['alcohol'] / (df_new['density'] + 1e-5)
#     df_new['acidity_per_pH'] = df_new['total_acidity'] / (df_new['pH'] + 1e-5)
#     numeric_cols = ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
#                     'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
#                     'pH', 'sulphates', 'alcohol']
#     df_new['feature_mean'] = df_new[numeric_cols].mean(axis=1)
#     df_new['feature_std'] = df_new[numeric_cols].std(axis=1)
#     df_new['feature_max'] = df_new[numeric_cols].max(axis=1)
#     df_new['feature_min'] = df_new[numeric_cols].min(axis=1)
#     df_new['feature_range'] = df_new['feature_max'] - df_new['feature_min']
#     df_new['acid_balance'] = df_new['fixed acidity'] / (df_new['volatile acidity'] + df_new['citric acid'] + 1e-5)
#     df_new['preservation_index'] = df_new['total sulfur dioxide'] * df_new['sulphates']
#     df_new['sweet_acid_ratio'] = df_new['residual sugar'] / (df_new['total_acidity'] + 1e-5)
#     df_new = df_new.replace([np.inf, -np.inf], np.nan)
#     df_new = df_new.fillna(df_new.median())
#     return df_new

# train_df_fe = create_features(train_df)
# test_df_fe = create_features(test_df)

# target = "quality"
# feature_cols = [col for col in train_df_fe.columns if col not in ['id', target]]

# X_train = train_df_fe[feature_cols]
# y_train = train_df_fe[target]
# X_test = test_df_fe[feature_cols]

# min_quality = int(y_train.min())
# max_quality = int(y_train.max())

# ridge_model = Ridge(alpha=1.0, random_state=42)
# ridge_model.fit(X_train, y_train)

# train_pred = ridge_model.predict(X_train)
# train_pred_clipped = np.clip(train_pred, min_quality, max_quality)
# train_pred_rounded = np.round(train_pred_clipped).astype(int)
# y_train_int = y_train.astype(int)

# rmse = np.sqrt(mean_squared_error(y_train, train_pred))
# mae = mean_absolute_error(y_train, train_pred)
# accuracy = accuracy_score(y_train_int, train_pred_rounded)
# f1_macro = f1_score(y_train_int, train_pred_rounded, average='macro')
# f1_weighted = f1_score(y_train_int, train_pred_rounded, average='weighted')

# test_predictions = ridge_model.predict(X_test)
# test_predictions_clipped = np.clip(test_predictions, min_quality, max_quality)
# test_predictions_int = np.round(test_predictions_clipped).astype(int)

# sample_sub_df[target] = test_predictions_int
# sample_sub_df.to_csv("submission_ridge.csv", index=False)
